# Phase 12 — Dekohärenz: ist das Antennen-Cluster kausal relevant?

Letzte offene Flagge aus Cell 27A. Die Router-Zeilen der 317 werden
orthogonalisiert (Kohärenz → 0, Normen erhalten); Kontrollarme: gleiche
Bewegungsdistanz mit zufälliger Störung, und dieselbe Operation auf der
Kontrollgruppe. Die 317-Rekrutierung am Köder-Token wird mitgemessen, damit
ein Kipp-Tod nicht mit einer verkappten Ablation verwechselt wird.
Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~8 min.

In [ ]:
# === Dekohaerenz-Test — ist das Antennen-Cluster GEFRAGT oder nur evident? =
# Cell 27A fand: die Router-Zeilen der 317 sind kohaerenter als Zufalls-
# Experten (+7.8%, p=0.0002). Offen blieb die Relevanz: etwas kann klar
# anders und trotzdem egal sein. Hier der Kausaltest - die Antennen der 317
# werden ORTHOGONALISIERT (Polarfaktor U@Vh, Zeilennormen erhalten): die
# Kohaerenz faellt per Konstruktion auf 0, die Zeilen bleiben gleich lang.
# Arme (N=32):  none | ortho-317 | rausch-317 (gleiche Bewegungs-Distanz,
# aber Kohaerenz bleibt - trennt 'Kohaerenz weg' von 'Gewichte bewegt') |
# ortho-Kontrolle (gleiche Operation auf der Zufalls-Kontrollgruppe).
# Mitgemessen: die 317-REKRUTIERUNG am Koeder-Token - bricht sie ein, war
# der Eingriff eine verkappte Maske und beweist nichts ueber Kohaerenz.
# Gewichte werden gesichert und am Ende (auch bei Fehler) wiederhergestellt.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
N_ANS=32; MAX_NEW=24; CHUNK=8; TOP_K=8; SEED=0
BANNED_JSON=globals().get("BANNED_JSON",
    (glob.glob("/content/drive/MyDrive/**/banned_experts.json",recursive=True) or [""])[0])
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def orthogonalize(R,eps=1e-9):
    """naechster zeilen-orthonormaler Satz (Polarfaktor U@Vh), Zeilennormen erhalten.
       Zerstoert die Paar-Kohaerenz per Konstruktion (cos -> 0)."""
    nrm=R.norm(dim=1,keepdim=True)
    U,S,Vh=torch.linalg.svd(R,full_matrices=False)
    Q=U@Vh
    return Q/Q.norm(dim=1,keepdim=True).clamp_min(eps)*nrm
def perturb_like(R,dist,seed,eps=1e-9):
    """zufaellige Stoerung mit VORGEGEBENEM Frobenius-Abstand, Zeilennormen erhalten.
       Kontrolle: gleiche Bewegung, aber ohne Kohaerenz-Abbau."""
    g=torch.Generator(device="cpu").manual_seed(seed)
    Nz=torch.randn(R.shape,generator=g).to(R.device,R.dtype)
    Nz=Nz/Nz.norm().clamp_min(eps)*dist
    nrm=R.norm(dim=1,keepdim=True); R2=R+Nz
    return R2/R2.norm(dim=1,keepdim=True).clamp_min(eps)*nrm
def coh(R,eps=1e-9):
    """medianer |cos| ueber alle Zeilenpaare"""
    Rn=R/R.norm(dim=1,keepdim=True).clamp_min(eps)
    M=(Rn@Rn.T).abs()
    iu=torch.triu_indices(R.shape[0],R.shape[0],offset=1)
    return float(M[iu[0],iu[1]].median())
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def verdict_deco(kn,kd,kr,kc,rec_n,rec_d,N,c_n=None,c_d=None,rec_frac=0.5,land=0.8):
    """kn none | kd Dekohaerenz-317 | kr Zufallsrichtung-317 | kc Dekohaerenz-Kontrolle
       rec_n/rec_d: 317-Rekrutierung am Koeder-Token vorher/nachher
       c_n/c_d: Kohaerenz vorher/nachher (Landungs-Pruefung)"""
    kill=lambda k:(k<kn and twoprop(k,N,kn,N)<0.05)
    if (c_n is not None and c_d is not None and c_d>land*c_n and not kill(kd)):
        return "EINGRIFF-SCHWACH"
    if not kill(kd): return "IRRELEVANT"
    if rec_d < rec_frac*rec_n: return "VERKAPPTE-MASKE"
    if kill(kr) or kill(kc): return "UNSPEZIFISCH"
    return "RELEVANT"
def tok_span(offs,c0,c1):
    return [i for i,(s,e) in enumerate(offs) if e>c0 and s<c1 and e>s]
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- Router + Gruppen ------------------------------------------
banned={int(l):sorted(v) for l,v in json.load(open(BANNED_JSON)).items()}
MLPM={}
for name,mod in model.named_modules():
    m=re.fullmatch(r"model\.layers\.(\d+)\.mlp",name)
    if m and hasattr(mod,"gate") and hasattr(mod.gate,"weight"): MLPM[int(m.group(1))]=mod
assert MLPM, "keine MoE-Bloecke gefunden"
E=MLPM[sorted(MLPM)[0]].gate.weight.shape[0]
rng=np.random.default_rng(SEED)
CTRL={l:sorted(rng.choice([e for e in range(E) if e not in banned.get(l,[])],
      size=len(banned.get(l,[])),replace=False).tolist()) for l in MLPM if banned.get(l)}
BVEC={}
for l in MLPM:
    v=torch.zeros(E,dtype=torch.bool)
    for e in banned.get(l,[]): v[e]=True
    BVEC[l]=v
BAK={l:MLPM[l].gate.weight.detach().clone() for l in MLPM}
DIST={l:float((orthogonalize(BAK[l].float()[banned[l]])-BAK[l].float()[banned[l]]).norm())
      for l in MLPM if len(banned.get(l,[]))>=2}   # Bewegungs-Distanz je Layer
print("MoE: %d Layer | %d Experten | banned gesamt %d"%(len(MLPM),E,sum(len(v) for v in banned.values())))
# ---------------- Eingriff --------------------------------------------------
def apply_deco(groups,mode):
    """groups: {layer:[ids]}; mode 'ortho' | 'noise' | None (Original)"""
    moved=[]
    for l in MLPM:
        W=BAK[l].float().clone()
        ids=groups.get(l,[]) if groups else []
        if mode is not None and len(ids)>=2:
            R=W[ids]
            if mode=="ortho":
                R2=orthogonalize(R); moved.append(float((R2-R).norm()))
            else:
                d=DIST.get(l,float((orthogonalize(R)-R).norm()))
                R2=perturb_like(R,d,SEED+l); moved.append(float((R2-R).norm()))
            W[ids]=R2
        MLPM[l].gate.weight.data.copy_(W.to(MLPM[l].gate.weight.dtype))
    return float(np.median(moved)) if moved else 0.0
def restore():
    for l in MLPM: MLPM[l].gate.weight.data.copy_(BAK[l])
def group_coh(groups=None):
    """mediane Paar-Kohaerenz der 317 ueber alle Layer (aktueller Gewichtsstand)"""
    vals=[]
    for l in MLPM:
        ids=(groups or banned).get(l,[])
        if len(ids)>=2: vals.append(coh(MLPM[l].gate.weight.detach().float()[ids]))
    return float(np.median(vals)) if vals else float("nan")
# ---------------- Rekrutierungs-Messung (Hooks auf die Router) --------------
ST={"n_ret":1,"rec_all":0.0,"n_all":0,"rec_dec":0.0,"n_dec":0}
def mk_hook(l):
    bvec=BVEC[l]
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if t.shape[-1]!=E: return out
        lg=t.reshape(-1,E); rows=int(lg.shape[0])
        if rows==ST["n_ret"] or rows%L!=0: return out          # nur volle Prefills
        ti=lg.topk(TOP_K,dim=-1).indices
        hb=bvec.to(ti.device)[ti.reshape(-1)].reshape(rows,TOP_K).sum(1).float()
        ST["rec_all"]+=float(hb.sum()); ST["n_all"]+=rows
        pos=torch.arange(rows,device=lg.device)%L
        msk=torch.isin(pos,DEC_T.to(lg.device))
        ST["rec_dec"]+=float(hb[msk].sum()); ST["n_dec"]+=int(msk.sum())
        return out
    return h
# ---------------- Prompt --------------------------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
prefix=think_prefix(TAB,"")
enc=tokenizer(prefix,return_offsets_mapping=True)
IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=tok_span(enc["offset_mapping"],c0,c1); DEC_T=torch.tensor(DEC)
print("Koeder-Span %s %r | Prompt %d Tokens"%(DEC,tokenizer.decode([IDS[i] for i in DEC]),L))
dev=model.device
ids_t=torch.tensor([IDS],device=dev)
hooks=[MLPM[l].gate.register_forward_hook(mk_hook(l)) for l in MLPM]
@torch.no_grad()
def run_arm(n,max_new):
    outs=[]
    ST.update(rec_all=0.0,n_all=0,rec_dec=0.0,n_dec=0)
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        ST["n_ret"]=b
        out=model.generate(ids_t.repeat(b,1),do_sample=True,temperature=1.0,top_p=1.0,
                           top_k=0,repetition_penalty=1.0,max_new_tokens=max_new,
                           pad_token_id=tokenizer.eos_token_id)
        outs+=[tokenizer.decode(o[L:],skip_special_tokens=True) for o in out]
    rec_all=ST["rec_all"]/max(ST["n_all"],1); rec_dec=ST["rec_dec"]/max(ST["n_dec"],1)
    return outs,rec_all,rec_dec
SW=("takeover","gloss","latin-switch(fr)")
ARMS=[("none",None,None),
      ("ortho-317",banned,"ortho"),
      ("rausch-317",banned,"noise"),
      ("ortho-Kontrolle",CTRL,"ortho")]
R={}
print("\nARME (N=%d):"%N_ANS)
try:
    for name,grp,mode in ARMS:
        mv=apply_deco(grp,mode)
        c317=group_coh()
        outs,ra,rd=run_arm(N_ANS,MAX_NEW)
        cls=[classify_answer(x) for x in outs]
        k=sum(1 for c in cls if c in SW)
        R[name]=(k,N_ANS,c317,ra,rd,dict(collections.Counter(cls)),mv)
        p,lo,hi=wilson(k,N_ANS)
        print("  %-16s rate=%5.1f%% [%4.1f,%4.1f] | Koh(317)=%.3f | Rekrut ges %.3f, Koeder %.3f | Bewegung %.2f  %s"
              %(name,100*p,100*lo,100*hi,c317,ra,rd,mv,R[name][5]))
finally:
    for h in hooks: h.remove()
    restore()
    ok=all(torch.equal(MLPM[l].gate.weight.data,BAK[l]) for l in MLPM)
    print("  (Gewichte wiederhergestellt: %s)"%("ja" if ok else "NEIN - Modell neu laden!"))
kn,_,c_n,_,rd_n,_,_=R["none"]; kd,_,c_d,_,rd_d,_,mv_d=R["ortho-317"]
kr=R["rausch-317"][0]; kc=R["ortho-Kontrolle"][0]
code=verdict_deco(kn,kd,kr,kc,rd_n,rd_d,N_ANS,c_n=c_n,c_d=c_d)
print("\n  Eingriff gelandet? Kohaerenz der 317: %.3f -> %.3f (Rausch-Arm %.3f, gleiche Bewegung)"%(
        c_n,c_d,R["rausch-317"][2]))
print("  p(deko-317 vs none)=%.4f | Rekrutierung am Koeder %.3f -> %.3f (%.0f%% erhalten)"
      %(twoprop(kd,N_ANS,kn,N_ANS),rd_n,rd_d,100*rd_d/max(rd_n,1e-9)))
print("\nVERDIKT:",end=" ")
if code=="RELEVANT":
    print("KOHAERENZ IST KAUSAL: die gemeinsame Richtung herauszuprojizieren toetet")
    print("  den Kipp (%d/32 -> %d/32), waehrend die 317 weiter rekrutiert werden"%(kn,kd))
    print("  (%.0f%% erhalten) und weder der distanzgleiche Rausch-Arm (%d/32) noch die"%(100*rd_d/max(rd_n,1e-9),kr))
    print("  orthogonalisierte Kontrollgruppe (%d/32) wirken. Das Cluster ist GEFRAGT."%kc)
elif code=="VERKAPPTE-MASKE":
    print("VERKAPPTE MASKE: der Kipp stirbt (%d/32), aber die Rekrutierung bricht mit"%kd)
    print("  ein (%.3f -> %.3f) - die Dekohaerenz wirkt als Auswahl-Ablation, nicht als"%(rd_n,rd_d))
    print("  Richtungs-Eingriff. Keine Aussage ueber die Kohaerenz moeglich.")
elif code=="UNSPEZIFISCH":
    print("UNSPEZIFISCH: auch der Rausch-Arm (%d/32) oder die Kontrollgruppe (%d/32)"%(kr,kc))
    print("  druecken den Kipp - jede gleich grosse Router-Stoerung wirkt, die")
    print("  Kohaerenz ist daran nichts Besonderes. Kein Beleg fuer Relevanz.")
elif code=="EINGRIFF-SCHWACH":
    print("EINGRIFF ZU SCHWACH: die Orthogonalisierung senkt die Kohaerenz kaum")
    print("  (%.3f -> %.3f) - unerwartet, Gewichtsschreibung pruefen."%(c_n,c_d))
else:
    print("IRRELEVANT: die Antennen lassen sich orthogonalisieren (Kohaerenz %.3f ->"%c_n)
    print("  %.3f), ohne dass der Kipp leidet (%d/32 vs. Baseline %d/32). Das Antennen-"%(c_d,kd,kn))
    print("  Cluster ist eine EIGENSCHAFT ohne kausale Rolle - evident, aber egal.")
    print("  Damit faellt die letzte gewichtsseitige Saeule; das Verhalten haengt an")
    print("  Inhalt und Position, nicht an der Verdrahtungs-Geometrie.")
DECO_RESULTS=dict(verdict=code,arms={k:(v[0],v[1],v[2],v[3],v[4],v[6]) for k,v in R.items()})
